# Delft: a first run

This tutorial downloads a ready case, the morning of Delft, runs it with one command and reads what the run wrote:
who took which mode and how long it took, how the iterations settled, the traffic on the links for cars, bikes and
walkers, one traveller's journey leg by leg, and the two pages a run can write, the map and the summary.

```bash
pip install openmobisim pandas pyarrow
```

openmobisim itself needs nothing else. A run writes its tables as Parquet files, and reading them in Python needs a
Parquet reader: here pandas with pyarrow ([Reading the results](../guides/results.md)).

## The case

The case is one of five published as a data record ([The cases](../guides/cases.md)). Its network and timetable are
real; its travellers are synthetic, not observed and not calibrated.

In [1]:
import os, urllib.request, zipfile

urllib.request.urlretrieve("https://zenodo.org/records/24585967/files/delft.zip", "delft.zip")
zipfile.ZipFile("delft.zip").extractall()
sorted(os.listdir("delft"))

['LICENCE.md',
 'gtfs.zip',
 'links.parquet',
 'nodes.parquet',
 'run.toml',
 'trips.parquet']

`nodes.parquet` and `links.parquet` are the network, `gtfs.zip` the timetable, `trips.parquet` the travellers and
`run.toml` the settings. In these cases each link is for one mode, so a street that cars, bikes and walkers all use
is three links.

In [2]:
import pandas as pd

links = pd.read_parquet("delft/links.parquet")
links.groupby("modes")["length_m"].agg(links="count", km=lambda metres: metres.sum() / 1000).round(0)

,links,km
modes,,
bike,20652,1240.0
car,12460,912.0
walk,55400,2253.0


Each traveller is one row: the two ends of the trip, the second of departure (seconds after midnight), a traveller
class and the modes open to that traveller.

In [3]:
travellers = pd.read_parquet("delft/trips.parquet")
travellers.head(3)

,trip_id,origin_lon_deg,origin_lat_deg,destination_lon_deg,destination_lat_deg,departure_s,class_id,modes
0,od0000001,4.390874,51.966004,4.403942,51.968401,30279,car_and_bike,car;bike;walk;transit
1,od0000010,4.406858,51.968952,4.378708,51.987099,30389,bike_only,bike;walk;transit
2,od0000011,4.403942,51.968401,4.378708,51.987099,30046,car_only,car;walk;transit


In [4]:
travellers.groupby(["class_id", "modes"]).size().rename("travellers").reset_index()

,class_id,modes,travellers
0,bike_only,bike;walk;transit,2663
1,car_and_bike,car;bike;walk;transit,5261
2,car_only,car;walk;transit,1264
3,from_outside_car,car,16544
4,from_outside_transit,transit,6325
5,from_outside_transit,transit;bike,89
6,goods,car,1505
7,no_vehicle,walk;transit,3998


## The run

`run_file` reads the settings file `run.toml` in the folder, builds the network, the timetable and the demand from
the files it names, and runs. The settings give the window, 07:30 to 09:30 (27 000 s to 34 200 s), and every
parameter at its value ([Parameters](../reference/parameters.md)).

In [5]:
import time
import openmobisim as oms

start = time.perf_counter()
run = oms.run_file("delft", out="delft-run")
print(f"run time {time.perf_counter() - start:.1f} s; ended by the stopping rule: {run.stopped_by_rule}")
sorted(os.listdir("delft-run"))

run time 3.9 s; ended by the stopping rule: True


['iterations.parquet',
 'links.parquet',
 'routes.parquet',
 'run_record.toml',
 'trips.parquet']

The run wrote four tables and its record. `run.trips`, `run.links`, `run.iterations`, `run.routes` and `run.record`
are their paths.

## Who took what, and how long it took

`trips.parquet` has one row per traveller: the mode taken and the travel time, door to door (the walk to and from
the network or a stop included).

In [6]:
trips = pd.read_parquet(run.trips)

def by_mode(trips):
    """Per mode: the travellers who took it, their share and their mean door-to-door time."""
    table = trips.groupby("mode").agg(travellers=("trip_id", "count"), mean_time_min=("travel_time_s", "mean"))
    table["mean_time_min"] /= 60
    table.insert(1, "share_pct", 100 * table["travellers"] / len(trips))
    return table.loc[["car", "bike", "walk", "transit"]].round(1)

by_mode(trips)

,travellers,share_pct,mean_time_min
mode,,,
car,22475,59.7,8.3
bike,4364,11.6,12.8
walk,1614,4.3,24.8
transit,9196,24.4,28.3


In [7]:
print("unfinished at 09:30:", (~trips["finished"]).sum())

unfinished at 09:30: 0


A trip that has not arrived when the window ends is unfinished and has no travel time; means are over finished
trips, so read them with the unfinished count ([The window and unfinished trips](../guides/window.md)). Here every
traveller arrives.

## How the iterations settled

Iteration 0 runs on free-flow times and the timetable: everyone chooses as if the roads were empty. From then on,
each iteration searches for better alternatives at the last loading's times, offers a share of the travellers a new
choice, loads, and measures. `iterations.parquet` has one row per iteration.

In [8]:
it = pd.read_parquet(run.iterations)
pd.DataFrame({
    "mean trip (min)": it["mean_trip_time_s"] / 60,
    "unfinished": it["unfinished"],
    "offered a new choice (%)": 100 * it["offered_share"],
    "switched (%)": 100 * it["switch_share"],
    "found a better alternative (%)": 100 * it["found_share"],
    "would change if asked (%)": 100 * it["would_change_share"],
}).round(2)

,mean trip (min),unfinished,offered a new choice (%),switched (%),found a better alternative (%),would change if asked (%)
0,21.37,4689,100.00,0.00,100.00,0.83
1,15.99,0,59.23,18.76,34.52,14.33
2,15.10,0,43.46,8.75,15.73,13.69
3,14.64,0,35.39,5.74,9.88,11.68
4,14.48,0,30.20,3.78,2.49,9.30
5,14.43,0,26.31,2.71,1.81,7.65
6,14.44,0,23.19,1.94,1.84,6.50
7,14.41,0,21.29,1.47,0.73,5.60
8,14.42,0,19.45,1.08,0.52,4.90
9,14.42,0,17.75,0.96,0.55,4.34


The run stops after an iteration when the last five mean trip times lie within 0.3 % of the latest and the search
found a better alternative for at most 0.5 % of the trips, or at the cap on iterations. Here the mean was steady
from iteration 9, and the found share fell under 0.5 % at iteration 10. The last column is reported and not judged;
at iteration 0 it is counted over sets that hold one route per mode
([The iteration and the stopping rule](../concepts/iteration.md)).

## The traffic on the links

`links.parquet` has one row per link and time bin (300 s here) that someone entered: the cars with the time they
spent on the link, the bikes and the walkers. Joined with the network's lengths it gives the distance travelled on
the network per mode. The walkers counted on links are those of walk trips; the walks to a stop or between a door
and the network are not on links.

In [9]:
bins = pd.read_parquet(run.links).merge(links[["link_id", "length_m", "car_speed_km_h", "road_class"]], on="link_id")
pd.Series({mode: (bins[column] * bins["length_m"]).sum() / 1000
           for mode, column in (("car", "cars"), ("bike", "bikes"), ("walk", "walkers"))}, name="km on the network").round(0)

car     187209.0
bike     12612.0
walk      3069.0
Name: km on the network, dtype: float64

Cars queue; bikes and walkers travel at free flow. A link's free-flow time is its length divided by its free-flow
speed, so the time the cars lost on a link is the time they spent there minus the cars times the free-flow time.

In [10]:
cars = bins[bins["cars"] > 0].copy()
cars["lost_s"] = cars["car_time_s"] - cars["cars"] * cars["length_m"] / (cars["car_speed_km_h"] / 3.6)
print(f"car delay: {cars['lost_s'].sum() / 3600:.0f} vehicle hours, "
      f"{100 * cars['lost_s'].sum() / cars['car_time_s'].sum():.1f} % of the cars' time on links")
worst = cars.groupby(["link_id", "road_class", "length_m"])["lost_s"].sum().div(3600).nlargest(5)
worst.rename("vehicle hours lost").reset_index().drop(columns="link_id").round(1)

car delay: 394 vehicle hours, 13.8 % of the cars' time on links


,road_class,length_m,vehicle hours lost
0,motorway_link,204.9,82.8
1,motorway_link,16.5,62.9
2,motorway_link,142.7,49.7
3,motorway_link,204.1,44.5
4,tertiary,232.8,37.4


## One traveller, leg by leg

`routes.parquet` holds what each traveller took at the last iteration, one row per leg. A transit journey is a walk
to a stop, its rides (with the walks between them) and a walk to the door; a car, bike or walk trip is a walk to the
network, its route as a list of links, and a walk to the door. Here is the first transit traveller with two rides.

In [11]:
routes = pd.read_parquet(run.routes)
rides = routes[routes["kind"] == "ride"].groupby("trip_id").size()
trip_id = rides[rides == 2].index[0]
trips[trips["trip_id"] == trip_id][["trip_id", "departure_s", "arrival_s", "travel_time_s"]].round(0)

,trip_id,departure_s,arrival_s,travel_time_s
5,od0000016,27049,30583.0,3534.0


In [12]:
legs = routes[routes["trip_id"] == trip_id]
legs[["leg", "kind", "seconds", "board_stop", "board_s", "alight_stop", "alight_s"]].round(0)

,leg,kind,seconds,board_stop,board_s,alight_stop,alight_s
15,0,walk_gap,67.0,NaN,NaN,NaN,NaN
16,1,ride,NaN,3989648,27768.0,3990650,28620.0
17,2,transfer_walk,15.0,NaN,NaN,NaN,NaN
18,3,ride,NaN,3428643,28800.0,3621208,29640.0
19,4,walk_gap,943.0,NaN,NaN,NaN,NaN


The traveller leaves the door at the departure second (seconds after midnight), walks to the stop and waits there
for the run of the journey chosen; that wait is part of the travel time. A ride has its two clock seconds, a walk
its duration.

## The map and the summary

A run can write two pages, each one self-contained HTML file that opens from disk with no server and no network.

In [13]:
run.map("delft-map.html")
run.summary("delft-summary.html")
for page in ("delft-map.html", "delft-summary.html"):
    print(f"{page}: {os.path.getsize(page) / 1e6:.1f} MB")

delft-map.html: 3.2 MB
delft-summary.html: 2.1 MB


![The map page of the run: Delft's roads, one ribbon per direction, coloured by volume over capacity](../_static/figures/delft_map.png)

The map page as it opens (the network is from OpenStreetMap, © OpenStreetMap contributors): road traffic by time
bin, with layers for transit lines, cyclists and walkers, the journeys between the busiest pairs of places, and, for
a link clicked, the pairs that use it. The summary page gives the run's numbers: the modes, when people left, the
roads, transit, bikes and walking, the iterations and the record ([The map and the summary](../guides/pages.md)).

## The record, and running again

`run_record.toml` is the record of what was run. Its first part is the settings the run ran with, every parameter at
the value used; its second part is what the run found: how it ended and the fingerprints of its inputs.

In [14]:
record = open(run.record).read()
print(record[record.index("window_end_used_s"):record.index("[result.transit.gtfs_report]")].strip())

window_end_used_s = 34200  # s: window_end_s, moved up to a whole link_bin when it was not one
iterations = 11
stopped_by_rule = true

[result.network]
source = "tables"
nodes = 21566
links = 88512
fingerprint = "3f87b0d8e1693b38"

[result.demand]
trips = 37649
fingerprint = "4ba8ec884eb6e048"

[result.transit]
stops = 1340
runs = 1707
fingerprint = "b78ffca832c14a3c"


The same inputs and settings give the same trips, links and routes tables, at any number of threads and over
repeated runs. `Run.open` opens a finished run from its folder without running anything, for instance to write its
pages later:

```python
run = oms.Run.open("delft-run")
```

To change a setting, edit `run.toml` (or copy the folder and edit the copy) and run again: the
[settings guide](../guides/settings.md) shows how. The [next tutorial](amsterdam.ipynb) builds a case step by step in
Python.